In [ ]:
from pathlib import Path

from czitools import read_tools
from czitools import metadata_tools as czimd

from pylibCZIrw import czi as pyczi

from aicspylibczi import CziFile

import dask.array as da
import numpy as np

import pandas as pd
from pandarallel import pandarallel

from tqdm.auto import tqdm

#pylibCZIrw completely abstracts away the subblock concept, which precludes me from accessing acquisition tiles

In [ ]:
filepath = Path('E:\PROJECTS\LSdata\JuliaCCI\Image_20.czi')

# get the metadata at once as one big class
mdata = czimd.CziMetadata(filepath)

# get only specific metadata
czi_dimensions = czimd.CziDimensions(filepath)
print("SizeS: ", czi_dimensions.SizeS)
print("SizeT: ", czi_dimensions.SizeT)
print("SizeZ: ", czi_dimensions.SizeZ)
print("SizeC: ", czi_dimensions.SizeC)
print("SizeY: ", czi_dimensions.SizeY)
print("SizeX: ", czi_dimensions.SizeX)

In [ ]:
with pyczi.open_czi(str(filepath)) as czidoc:
    plane = {'C': 0, 'Z': 0, 'T': 0}
    frame_0 = czidoc.read(plane=plane)
    # Get the total bounding box for all scenes
    total_bounding_rectangle = czidoc.total_bounding_rectangle


assert total_bounding_rectangle.h == frame_0.shape[0], 'sanity check failed in x dim'
assert total_bounding_rectangle.w == frame_0.shape[1], 'sanity check failed in y dim'
# if it fails look at notes in next cell

In [ ]:
czi = CziFile(filepath)
print(czi.is_mosaic())

In [ ]:
# Get the number of tiles
sb_meta = czi.read_subblock_metadata(C=0, T=0, Z=200)

In [ ]:
from matplotlib import pyplot as plt
# importing gaussian filter from scikit-image
from skimage.filters import gaussian

In [ ]:
from skimage.transform import resize

def downsample_blur_upsample(image, sigma, scale=0):
    if scale == 0:
        scale = max(1, sigma / 2)
    small_size = tuple(int(s / scale) for s in image.shape)
    small_image = resize(image, small_size, anti_aliasing=True, preserve_range=True)
    blurred_small = gaussian(small_image, sigma=sigma/scale, preserve_range=True)
    return resize(blurred_small, image.shape, anti_aliasing=False, preserve_range=True)

def get_pixeld_dtype(aics_czi):
    # Get the pixel type, 
    pixel_type = aics_czi.pixel_type

    if pixel_type == 'gray16':
        imgdtype = np.uint16
    elif pixel_type == 'gray8':
        imgdtype = np.uint8
    else:
        # wrtie error message return empty array
        imgdtype = []

    return imgdtype

In [ ]:
def fill_correct_tile(aics_czi, sigma=400, scale=10, C=0, T=0, Z=0):
    mbbox = aics_czi.get_mosaic_bounding_box()

    # Get the pixel type, 
    pixel_type = aics_czi.pixel_type

    if pixel_type == 'gray16':
        imgdtype = np.uint16
    elif pixel_type == 'gray8':
        imgdtype = np.uint8
    else:
        # wrtie error message return empty array
        return


    img = np.zeros((mbbox.h, mbbox.w), dtype=imgdtype)

    mosaic_info = czi.get_all_mosaic_tile_bounding_boxes(C=C, T=T, Z=Z)
    #x_coords = []
    #y_coords = []

    for m_idx, (scene_index, bbox) in enumerate(mosaic_info.items()):
        tile_img, tile_shp = czi.read_image(M=m_idx, C=C, T=T, Z=Z)
        tile_img = tile_img.squeeze()
        bg = downsample_blur_upsample(tile_img, sigma=sigma, scale=scale)

        tile_corr = tile_img - bg
        tile_corr[tile_corr<0] = 0
        tile_corr = tile_corr.astype(img.dtype)

        img[bbox.y:bbox.y+bbox.h, bbox.x:bbox.x+bbox.w] = tile_corr
        #print(m_idx)
        #x_coords.append(bbox.x)
        #y_coords.append(bbox.y)

    return img



In [ ]:
img_plane = fill_correct_tile(czi, sigma=400, scale=10, Z=200)
plt.imshow(img_plane)
plt.colorbar()
plt.clim(0, 500)

In [ ]:
# get an object containing only the dimension information
czi_scale = czimd.CziScaling(filepath)
x_pix_size = czi_scale.X
y_pix_size = czi_scale.Y
z_pix_size = czi_scale.Z
print(f'pixel size in microns (x, y, z): {x_pix_size:.3f}, {y_pix_size:.3f}, {z_pix_size:.3f}')

In [ ]:
#ro = 1.2
#r1 = 2.4
#r2 = 4.8
#r3 = 9.6 Then it becomes almost isotropic at this stage

In [ ]:
def optimal_size(current_size, res_levels):
    # helper function that asses the best size given the desired resolution level
    
    div_factor = np.power(2,res_levels)
    rem = np.remainder(current_size, div_factor)

    print(f'current size: {current_size}, factor: {div_factor}, reminder: {rem}')

    if rem > 0:
        extra = div_factor-rem
    else:
        extra = 0

    print(f'we need to add: {extra}, so new size is: {current_size+extra}')

    return current_size+extra


In [ ]:
res_levels_x = 4
res_levels_y = 4
res_levels_z = 1

x_size = czi_dimensions.SizeX
y_size = czi_dimensions.SizeY
z_size = czi_dimensions.SizeZ

total_x = int(optimal_size(czi_dimensions.SizeX, res_levels_x))
total_y = int(optimal_size(czi_dimensions.SizeY , res_levels_y))
total_z = int(optimal_size(czi_dimensions.SizeZ, res_levels_z))


In [ ]:
import zarr
def rm_tree(pth):
    pth = Path(pth)
    for child in pth.glob('*'):
        if child.is_file():
            child.unlink()
        else:
            rm_tree(child)
    pth.rmdir()

ch = 0
data_path = filepath.parent
czi_name = filepath.name
zarr_name = czi_name.split('.czi')
zarr_name = zarr_name[0] + '_ch' +ch.__str__()+'.zarr'

z0_path = data_path.joinpath(zarr_name)

print(z0_path)

In [ ]:
z_dtype = get_pixeld_dtype(czi)

if z0_path.exists():
  rm_tree(z0_path)
  
store = zarr.DirectoryStore(z0_path)
chunk_size = 512
z = zarr.creation.open_array(store=store, mode='a', shape=(total_z, total_y, total_x), chunks=(1, chunk_size,chunk_size), dtype=z_dtype)
z

In [ ]:
# Initialize an empty dictionary with lists for each key
info = {
    'M': [],
    'C': [],
    'T': [],
    'Z': [],
    'x': [],
    'y': [],
    'w': [],
    'h': [],
}


for z_idx in range(z_size):
    mosaic_info = czi.get_all_mosaic_tile_bounding_boxes(C=0, T=0, Z=z_idx)

    for m_idx, (scene_index, bbox) in enumerate(mosaic_info.items()):

        info['M'].append(m_idx)
        info['C'].append(0)
        info['T'].append(0)
        info['Z'].append(z_idx)
        info['x'].append(bbox.x)
        info['y'].append(bbox.y)
        info['w'].append(bbox.w)
        info['h'].append(bbox.h)

m_df = pd.DataFrame(info)
m_df.sample(10)

In [ ]:
# dynamically fill in values
from functools import partial
from multiprocess import Pool
from aicspylibczi import CziFile



def get_pixeld_dtype(aics_czi):
    from numpy import uint8, uint16
    # Get the pixel type, 
    pixel_type = aics_czi.pixel_type

    if pixel_type == 'gray16':
        imgdtype = uint16
    elif pixel_type == 'gray8':
        imgdtype = uint8
    else:
        # wrtie error message return empty array
        imgdtype = []

    return imgdtype


def correct_write(m_row, zarr_store, filepath, sigma=400, scale=10):
    from aicspylibczi import CziFile
    from skimage.transform import resize
    from skimage.filters import gaussian

    def downsample_blur_upsample(image, sigma, scale=0):
        if scale == 0:
            scale = max(1, sigma / 2)
        small_size = tuple(int(s / scale) for s in image.shape)
        small_image = resize(image, small_size, anti_aliasing=True, preserve_range=True)
        blurred_small = gaussian(small_image, sigma=sigma/scale, preserve_range=True)
        return resize(blurred_small, image.shape, anti_aliasing=False, preserve_range=True)

    aics_czi =  CziFile(filepath)
    tile_img, tile_shp = aics_czi.read_image(M=m_row['M'], C=m_row['C'], T=m_row['T'], Z=m_row['Z'])
    tile_img = tile_img.squeeze()

    bg = downsample_blur_upsample(tile_img, sigma=sigma, scale=scale)

    tile_corr = tile_img - bg
    tile_corr[tile_corr<0] = 0
    tile_corr = tile_corr.astype(tile_img.dtype)
    zarr_store[m_row['Z'], m_row['y']:m_row['y']+m_row['h'], m_row['x']:m_row['x']+m_row['w']] = tile_corr


# Partial function with the fixed zarr_store parameter
correct_write_partial = partial(correct_write, zarr_store=z,filepath=filepath, sigma = 400, scale = 10)

# Prepare a sequence of rows from the DataFrame
rows = [row for _, row in m_df.iterrows()]



for index, m_row in m_df.iterrows():
    print(f"Index: {index}")

    correct_write_partial(m_row)

    if index > 5:
        break


In [ ]:
unique_M = set(m_df['M'])
unique_M

for M_index in unique_M:
    print(M_index)
    M_list = [item for item in rows if item['M'] == M_index]

    # Use tqdm to wrap the list of rows for the progress bar
    with Pool(10) as pool:
        for _ in tqdm(pool.imap(correct_write_partial, M_list), total=len(M_list)):
            pass


In [ ]:
import dask.array as da
# like numpy.mean, but maintains dtype, helper function
def mean_dtype(arr, **kwargs):
    return np.mean(arr, **kwargs).astype(arr.dtype)

In [ ]:
# it is still not quite clear to me why, but we need to rechunk de data at this stage
# if not zarr writting later on will fail
d0 = da.from_zarr(store).rechunk((64,512,512))
d0

In [ ]:
from dask.diagnostics import ProgressBar
ProgressBar().register()

min_dask_val = d0.min()
max_dask_val = d0.max()
print(min_dask_val.compute())
print(max_dask_val.compute())

In [ ]:
# get an object containing only the dimension information
czi_scale = czimd.CziScaling(filepath)
x_pix_size = czi_scale.X
y_pix_size = czi_scale.Y
z_pix_size = czi_scale.Z
print(f'pixel size in microns (x, y, z): {x_pix_size:.3f}, {y_pix_size:.3f}, {z_pix_size:.3f}')

In [ ]:
# only rescale in x-y
# scale in z y x
d1_s = (1,2,2)
d1 = da.coarsen(mean_dtype, d0, {0:d1_s[0],1:d1_s[1],2:d1_s[2]}).rechunk((64,512,512))
# approaching isotorpic, only scale in x-y
d2_s = (1,4,4)
d2 = da.coarsen(mean_dtype, d0, {0:d2_s[0],1:d2_s[1],2:d2_s[2]}).rechunk((64,256,256))
# close to isotropic now scale for fast 3D rendering
d3_s = (1,8,8)
d3 = da.coarsen(mean_dtype, d0, {0:d3_s[0],1:d3_s[1],2:d3_s[2]}).rechunk((64,64,64))
# one further scale for fast 3D rendering
#d4_s = (1,16,16)
#d4 = da.coarsen(mean_dtype, d0, {0:d4_s[0],1:d4_s[1],2:d4_s[2]}).rechunk((64,64,64))
#d4

In [ ]:
from ome_zarr.io import parse_url
from ome_zarr.writer import write_multiscale
from ome_zarr.writer import write_multiscales_metadata

In [ ]:
# I can probably build this programmatically, for the moment I take a shortcut. 
# This assumes an image with full resolution and one downscale by 2x2
initial_pix_unit = 'micrometer'
coordtfs = [
        [{'type': 'scale', 'scale': [z_pix_size, y_pix_size, x_pix_size]},
         {'type': 'translation', 'translation': [0, 0, 0]}],

        [{'type': 'scale', 'scale': [z_pix_size*d1_s[0], y_pix_size*d1_s[1], x_pix_size*d1_s[2]]},
         {'type': 'translation', 'translation': [0, 0, 0]}],

        [{'type': 'scale', 'scale': [z_pix_size*d2_s[0], y_pix_size*d2_s[1], x_pix_size*d2_s[2]]},
         {'type': 'translation', 'translation': [0, 0, 0]}],

        [{'type': 'scale', 'scale': [z_pix_size*d3_s[0], y_pix_size*d3_s[1], x_pix_size*d3_s[2]]},
         {'type': 'translation', 'translation': [0, 0, 0]}],

        ]
axes = [{'name': 'z', 'type': 'space', 'unit': initial_pix_unit},
        {'name': 'y', 'type': 'space', 'unit': initial_pix_unit},
        {'name': 'x', 'type': 'space', 'unit': initial_pix_unit}]

In [ ]:
# Open the zarr group manually

omezarr_name = czi_name.split('.czi')
omezarr_name = omezarr_name[0] + '_ch' +ch.__str__()+'.ome.zarr'

path = data_path.joinpath(omezarr_name)

if path.exists():
  rm_tree(path)

store = parse_url(path, mode='w').store
root = zarr.group(store=store)

# Use OME write multiscale;
write_multiscale([d0, d1, d2, d3],
        group=root, axes=axes, coordinate_transformations=coordtfs
        )
# add omero metadata: the napari ome-zarr plugin uses this to pass rendering
# options to napari.
root.attrs['omero'] = {
        'channels': [{
                'color': 'ffffff',
                'label': 'ch'+ch.__str__(),
                'active': True,
                }]
        }

In [ ]:
z.shape
plt.colorbar()
plt.clim(0, 500)

# In case you want to inspect back in ZEN

In [ ]:
newczi = Path("./"+filepath.stem+".czi")
print(newczi)
pix_size ={
    'Value':czi_scale.X,
    'Unit':'µm',
    'Axial':czi_scale.Z
}
print(f"the pixel size in the plane is: {pix_size['Value']} {pix_size['Unit']}")
print(f"the pixel size axial is: {pix_size['Axial']} {pix_size['Unit']}")


In [ ]:
assert pix_size['Unit'] == "µm", "unit not in microns"

with pyczi.create_czi(newczi, exist_ok=True) as czidoc_w:
    # loop over z-planes and channels
    for frame in range(z.shape[0]):
        print(frame)
        # this must be in (Y,X,1)
        tmp_plane = z[frame,:,:].squeeze()
        czidoc_w.write(data=tmp_plane[...,np.newaxis],
                       plane={"Z": frame})

        # write the document title, channel names, custom attributes and XYZ scaling to the CZI file
    czidoc_w.write_metadata(
        document_name=data_path.stem,
        channel_names={0: "White"},
        scale_x=float(pix_size['Value']) * 10 ** -6,
        scale_y=float(pix_size['Value']) * 10 ** -6,
        scale_z=float(pix_size['Axial']) * 10 ** -6,
    )

In [ ]:
if z0_path.exists():
  rm_tree(z0_path)